# Carbon Emission Prediction Project

This notebook cleans the supplied dataset, saves a separate model-ready dataset, trains regression models, evaluates them, and identifies the most influential features. The original CSV is never changed.

## What this notebook does

1. Loads and audits the raw data.
2. Cleans values needed for modelling.
3. Converts multi-value fields into separate 0/1 columns.
4. One-hot encodes ordinary categorical columns.
5. Saves `Carbon Emission - LMS Model Ready.csv`.
6. Compares Ridge Regression, Random Forest, and Gradient Boosting.
7. Reports MAE, RMSE, R-squared, and feature importance.

In [1]:
# Run this once if your environment does not already have the packages.
# %pip install pandas numpy matplotlib seaborn scikit-learn

from pathlib import Path
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
RAW_FILE = Path('Carbon Emission - LMS.csv')
MODEL_READY_FILE = Path('Carbon Emission - LMS Model Ready.csv')

## 1. Load and audit the raw data

This checks the shape, data types, missing values, and a small sample before we make any changes.

In [ ]:
df = pd.read_csv(RAW_FILE)
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
display(df.head())
display(pd.DataFrame({'data_type': df.dtypes.astype(str), 'missing_values': df.isna().sum(), 'unique_values': df.nunique()}))

## 2. Quick exploratory analysis

The charts below help you understand the prediction target and compare average emissions across important lifestyle categories.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=df, x='CarbonEmission', bins=40, kde=True, ax=axes[0], color='#2a9d8f')
axes[0].set_title('Distribution of Carbon Emission')

transport_means = df.groupby('Transport', as_index=False)['CarbonEmission'].mean().sort_values('CarbonEmission', ascending=False)
sns.barplot(data=transport_means, x='Transport', y='CarbonEmission', ax=axes[1], color='#457b9d')
axes[1].set_title('Average Carbon Emission by Transport')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

for column in ['Diet', 'Heating Energy Source', 'Frequency of Traveling by Air', 'Energy efficiency']:
    print(f'\nAverage emissions by {column}:')
    display(df.groupby(column)['CarbonEmission'].mean().sort_values(ascending=False).round(2).to_frame())

## 3. Create a model-ready dataset

`Vehicle Type` is blank for records without a private vehicle. A blank is meaningful here, so it becomes `No vehicle` rather than being guessed or removed.

`Recycling` and `Cooking_With` contain lists in text form. We convert each possible item into its own 0/1 column. All remaining categories are one-hot encoded. This creates numeric input columns that regression models can use directly. No outliers are automatically deleted; they should be examined and justified in your report before removal.

In [ ]:
clean_df = df.copy()

# Clean text spacing and give blank vehicle types a meaningful label.
text_columns = clean_df.select_dtypes(include='object').columns
clean_df[text_columns] = clean_df[text_columns].apply(lambda col: col.str.strip())
clean_df['Vehicle Type'] = clean_df['Vehicle Type'].replace('', np.nan).fillna('No vehicle')

def parse_list(value):
    """Safely convert strings such as ['Paper', 'Glass'] into Python lists."""
    if isinstance(value, list):
        return value
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return []

def expand_multivalue_column(data, column, prefix):
    lists = data[column].apply(parse_list)
    all_items = sorted({item for row in lists for item in row})
    expanded = pd.DataFrame(
        {f'{prefix}{item}': lists.apply(lambda row: int(item in row)) for item in all_items},
        index=data.index,
    )
    return expanded

recycling_features = expand_multivalue_column(clean_df, 'Recycling', 'Recycling_')
cooking_features = expand_multivalue_column(clean_df, 'Cooking_With', 'Cooking_')
clean_df = clean_df.drop(columns=['Recycling', 'Cooking_With'])
clean_df = pd.concat([clean_df, recycling_features, cooking_features], axis=1)

target_column = 'CarbonEmission'
categorical_columns = clean_df.select_dtypes(include='object').columns.tolist()
model_ready_df = pd.get_dummies(clean_df, columns=categorical_columns, dtype=int)
model_ready_df.to_csv(MODEL_READY_FILE, index=False)

print(f'Model-ready dataset saved to: {MODEL_READY_FILE.resolve()}')
print(f'Model-ready shape: {model_ready_df.shape}')
print(f'Missing values after preprocessing: {model_ready_df.isna().sum().sum()}')
display(model_ready_df.head())

## 4. Split the data

We keep 20% of the data unseen until evaluation. This gives a fair estimate of performance on new households. `X` contains predictors and `y` contains the carbon-emission target.

In [ ]:
X = model_ready_df.drop(columns=target_column)
y = model_ready_df[target_column]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f'Training rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')
print(f'Predictor columns: {X.shape[1]}')

## 5. Train and compare models

Ridge is a linear baseline and needs scaled numeric inputs. Random Forest and Gradient Boosting can capture non-linear relationships. Use the same test set for every model so the comparison is fair. Lower MAE/RMSE and higher R-squared are better.

In [ ]:
models = {
    'Ridge Regression': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Random Forest': RandomForestRegressor(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = []
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_models[name] = model
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, predictions),
        'RMSE': root_mean_squared_error(y_test, predictions),
        'R_squared': r2_score(y_test, predictions),
    })

results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
display(results_df.style.format({'MAE': '{:,.2f}', 'RMSE': '{:,.2f}', 'R_squared': '{:.3f}'}))
best_model_name = results_df.loc[0, 'Model']
best_model = trained_models[best_model_name]
print(f'Best model by RMSE: {best_model_name}')

## 6. Validate the best model with cross-validation

Cross-validation trains the selected model on several different splits. It is more reliable than reporting one train/test split alone. Negative MAE is returned by scikit-learn, so we multiply by -1 to present it as a normal positive error.

In [ ]:
cv_mae = -cross_val_score(best_model, X, y, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
print(f'3-fold MAE: {cv_mae.mean():,.2f} ± {cv_mae.std():,.2f}')

## 7. Interpret feature importance

Tree models provide `feature_importances_`. If the best model is Ridge, we use the absolute coefficient size instead. Importance shows which variables the model used most; it does not prove that changing one feature causes emissions to change.

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=X.columns)
elif best_model_name == 'Ridge Regression':
    importance = pd.Series(np.abs(best_model.named_steps['model'].coef_), index=X.columns)
else:
    raise ValueError('Add permutation importance for this model type.')

top_features = importance.sort_values(ascending=False).head(15)
display(top_features.to_frame('importance'))
plt.figure(figsize=(10, 7))
sns.barplot(x=top_features.values, y=top_features.index, color='#e76f51')
plt.title(f'Top 15 feature importances: {best_model_name}')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 8. What to write in your final report

- State that the raw file had no missing values, but required preprocessing for categorical and list-style fields.
- Explain that blank vehicle types were converted to `No vehicle` because they indicate no private vehicle.
- Include the model comparison table and select the model with the lowest RMSE.
- Discuss the top features from the importance chart.
- Give recommendations supported by the results, for example cleaner transport, lower vehicle use, fewer flights, or more energy-efficient household behavior.
- Do not claim causation from feature importance alone.